# C_Silver - CDF + limpieza + estandarización

Silver consume exclusivamente las tablas Bronze mediante Change Data Feed (CDF).
Se conservan los cambios insert/update_postimage y se generan tablas Delta Silver.


In [0]:

from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}"
)


# ============================================================
# HABILITAR CDF EN BRONZE
# ============================================================

BRONZE_TABLES = [
    "calendar",
    "products",
    "customers",
    "stores",
    "sales"
]

for table_name in BRONZE_TABLES:

    full_table = f"{BRONZE_SCHEMA}.{table_name}"

    spark.sql(f"""
        ALTER TABLE {full_table}
        SET TBLPROPERTIES (
            delta.enableChangeDataFeed = true
        )
    """)

print(
    "CDF habilitado correctamente en las tablas Bronze"
)


# ============================================================
# FUNCIÓN PARA LEER CDF
# ============================================================

def read_cdf(table_name):

    return (
        spark.read
        .format("delta")
        .option(
            "readChangeFeed",
            "true"
        )
        .option(
            "startingVersion",
            0
        )
        .table(table_name)
        .filter(
            F.col("_change_type").isin(
                "insert",
                "update_postimage"
            )
        )
        .drop(
            "_change_type",
            "_commit_version",
            "_commit_timestamp"
        )
    )


# ============================================================
# FUNCIÓN PARA ESCRIBIR SILVER
# ============================================================

def write_table(df, table_name):

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(table_name)
    )


# ============================================================
# CALENDAR
# ============================================================

calendar = read_cdf(
    f"{BRONZE_SCHEMA}.calendar"
)

df_calendar_silver = (
    calendar
    .dropDuplicates()
    .dropna(
        subset=["date"]
    )
    .withColumn(
        "date",
        F.to_date("date")
    )
    .withColumn(
        "year",
        F.col("year").cast("int")
    )
    .withColumn(
        "month",
        F.col("month").cast("int")
    )
    .withColumn(
        "day",
        F.col("day").cast("int")
    )
    .withColumn(
        "week",
        F.col("week").cast("int")
    )
    .withColumn(
        "day_of_week",
        F.col("day_of_week").cast("int")
    )
)

write_table(
    df_calendar_silver,
    f"{SILVER_SCHEMA}.calendar"
)


# ============================================================
# PRODUCTS
# ============================================================

products = read_cdf(
    f"{BRONZE_SCHEMA}.products"
)

df_products_silver = (
    products
    .dropDuplicates()
    .dropna(
        subset=["product_id"]
    )
    .withColumn(
        "cocoa_percent",
        F.col("cocoa_percent").cast("int")
    )
    .withColumn(
        "weight_g",
        F.col("weight_g").cast("int")
    )
)

write_table(
    df_products_silver,
    f"{SILVER_SCHEMA}.products"
)


# ============================================================
# CUSTOMERS
# ============================================================

customers = read_cdf(
    f"{BRONZE_SCHEMA}.customers"
)

df_customers_silver = (
    customers
    .dropDuplicates()
    .dropna(
        subset=["customer_id"]
    )
    .withColumn(
        "age",
        F.col("age").cast("int")
    )
    .withColumn(
        "loyalty_member",
        F.col("loyalty_member").cast("int")
    )
    .withColumn(
        "join_date",
        F.to_date("join_date")
    )
)

write_table(
    df_customers_silver,
    f"{SILVER_SCHEMA}.customers"
)


# ============================================================
# STORES
# ============================================================

stores = read_cdf(
    f"{BRONZE_SCHEMA}.stores"
)

df_stores_silver = (
    stores
    .dropDuplicates()
    .dropna(
        subset=["store_id"]
    )
)

write_table(
    df_stores_silver,
    f"{SILVER_SCHEMA}.stores"
)


# ============================================================
# SALES
# ============================================================

sales = read_cdf(
    f"{BRONZE_SCHEMA}.sales"
)

df_sales_silver = (
    sales
    .dropDuplicates()
    .dropna(
        subset=[
            "order_id",
            "order_date",
            "product_id",
            "store_id",
            "customer_id"
        ]
    )
    .withColumn(
        "order_date",
        F.to_date("order_date")
    )
    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    )
    .withColumn(
        "unit_price",
        F.col("unit_price").cast("double")
    )
    .withColumn(
        "discount",
        F.col("discount").cast("double")
    )
    .withColumn(
        "revenue",
        F.col("revenue").cast("double")
    )
    .withColumn(
        "cost",
        F.col("cost").cast("double")
    )
    .withColumn(
        "profit",
        F.col("profit").cast("double")
    )
)

write_table(
    df_sales_silver,
    f"{SILVER_SCHEMA}.sales"
)


print(
    "SILVER - CDF + limpieza + estandarización finalizados correctamente"
)



CDF habilitado correctamente en las tablas Bronze
SILVER - CDF + limpieza + estandarización finalizados correctamente


In [0]:
%sql
SELECT 'calendar' AS tabla, COUNT(*) AS total_filas FROM workspace.silver.calendar
UNION ALL
SELECT 'products' AS tabla, COUNT(*) AS total_filas FROM workspace.silver.products
UNION ALL
SELECT 'customers' AS tabla, COUNT(*) AS total_filas FROM workspace.silver.customers
UNION ALL
SELECT 'stores' AS tabla, COUNT(*) AS total_filas FROM workspace.silver.stores
UNION ALL
SELECT 'sales' AS tabla, COUNT(*) AS total_filas FROM workspace.silver.sales;



tabla,total_filas
calendar,731
products,200
customers,150021
stores,100
sales,1000000


In [0]:
%sql
SHOW TBLPROPERTIES workspace.bronze.customers;

key,value
clusterByAuto,true
clusteringColumns,[]
delta.checkpointPolicy,v2
delta.enableChangeDataFeed,true
delta.enableDeletionVectors,true
delta.enableInCommitTimestamps,true
delta.enableRowTracking,true
delta.feature.appendOnly,supported
delta.feature.changeDataFeed,supported
delta.feature.clustering,supported


In [0]:
%sql
SELECT
    customer_id,
    age,
    gender,
    loyalty_member,
    join_date
FROM workspace.bronze.customers
WHERE source_file LIKE '%customers_y.csv';

customer_id,age,gender,loyalty_member,join_date
C000015,15,Male,0,2023-06-07
C000016,16,Male,0,2025-12-22


In [0]:
%sql
UPDATE workspace.bronze.customers
SET age =18
WHERE customer_id = 'C000015';

num_affected_rows
4


In [0]:
%sql
SELECT
    customer_id,
    age,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes(
    'workspace.bronze.customers',
    0
)
WHERE customer_id = 'C000015'
ORDER BY _commit_version DESC
LIMIT 2;

customer_id,age,_change_type,_commit_version,_commit_timestamp
C000015,18,update_preimage,29,2026-11-26T00:52:06.680Z
C000015,18,update_postimage,29,2026-11-26T00:52:06.680Z
